In [1]:
!pip install -q ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 22.8 MB/s eta 0:00:00


In [2]:
from ultralytics import YOLO
import torch

print("GPU:", torch.cuda.get_device_name(0))

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
GPU: Tesla T4


# Sửa file data.yaml

In [3]:
import yaml

# ===== ĐƯỜNG DẪN =====
YAML_PATH = "/kaggle/input/datasets/nguyenviet2709/data-yolo-v1/dataset_yolo/data.yaml"

# đọc file yaml
with open(YAML_PATH, "r") as f:
    data = yaml.safe_load(f)

print("Trước khi sửa:")
print(data)

Trước khi sửa:
{'path': '/kaggle/working/dataset_yolo', 'train': 'images/train', 'val': 'images/val', 'test': 'images/test', 'names': {0: 'ant', 1: 'butterfly', 2: 'cockroach', 3: 'dragonfly', 4: 'fly', 5: 'grasshopper', 6: 'honeybee', 7: 'ladybug', 8: 'mosquito', 9: 'spider'}}


In [4]:
# ===== PATH ĐÚNG =====
NEW_PATH = "/kaggle/input/datasets/nguyenviet2709/data-yolo-v1/dataset_yolo"

# sửa path
data["path"] = NEW_PATH

# đảm bảo có test (nếu bạn có)
data["train"] = "images/train"
data["val"]   = "images/val"
data["test"]  = "images/test"

# lưu sang working (QUAN TRỌNG)
FIXED_YAML_PATH = "/kaggle/working/data_fixed.yaml"

with open(FIXED_YAML_PATH, "w") as f:
    yaml.dump(data, f)

print("\nSau khi sửa:")
print(data)

print("\nĐã lưu file mới tại:", FIXED_YAML_PATH)


Sau khi sửa:
{'path': '/kaggle/input/datasets/nguyenviet2709/data-yolo-v1/dataset_yolo', 'train': 'images/train', 'val': 'images/val', 'test': 'images/test', 'names': {0: 'ant', 1: 'butterfly', 2: 'cockroach', 3: 'dragonfly', 4: 'fly', 5: 'grasshopper', 6: 'honeybee', 7: 'ladybug', 8: 'mosquito', 9: 'spider'}}

Đã lưu file mới tại: /kaggle/working/data_fixed.yaml


# Model

In [5]:
MODEL_NAME = "yolo11m.pt"
model = YOLO(MODEL_NAME)

# Train

In [6]:
results = model.train(
    data="/kaggle/working/data_fixed.yaml",

    # ===== Training =====
    epochs=120,
    patience=25,        # 🔥 Early stopping

    # ===== Image =====
    imgsz=640,

    # ===== Batch =====
    batch=16,           # nếu GPU đủ RAM có thể tăng 24 hoặc 32

    # ===== GPU =====
    device=0,           # dùng 1 GPU (ổn định nhất)

    # ===== Performance =====
    workers=4,
    amp=True,           # mixed precision → nhanh hơn

    # ===== Augmentation =====
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    degrees=5,
    translate=0.1,
    scale=0.5,
    fliplr=0.5,
    mosaic=1.0,         # mạnh nhưng nên giữ
    mixup=0.1,

    # ===== Regularization =====
    dropout=0.0,

    # ===== Logging =====
    project="/kaggle/working/yolo_train",
    name="insect_yolo11m_opt",
    save=True,
    verbose=True
)

Ultralytics 8.4.38 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/data_fixed.yaml, degrees=5, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=120, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.1, mode=train, model=yolo11m.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=insect_yolo11m_opt, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, pa

libpng warning: gAMA: out of place


      1/120      8.21G      1.542      2.538      1.648          6        640: 100% ━━━━━━━━━━━━ 188/188 1.5it/s 2:02
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 7% ╸─────────── 2/27 1.0s/it 3.7s<26.2s

libpng warning: iCCP: profile 'ICC Profile': 0h: PCS illuminant is not D50


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 27/27 1.2s/it 33.0s
                   all        851       2217      0.356      0.331      0.274      0.146

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      2/120      7.92G      1.567      2.087      1.651         74        640: 14% ━╸────────── 26/188 1.6it/s 16.4s<1:39

...

     88/120      8.28G      0.883     0.7077      1.187          8        640: 100% ━━━━━━━━━━━━ 188/188 1.7it/s 1:48
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 7% ╸─────────── 2/27 1.7it/s 0.6s<14.7s

libpng warning: iCCP: profile 'ICC Profile': 0h: PCS illuminant is not D50


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 27/27 2.7it/s 10.2s
                   all        851       2217      0.827      0.704      0.763      0.528
EarlyStopping: Training stopped early as no improvement observed in last 25 epochs. Best results observed at epoch 63, best model saved as best.pt.
To update EarlyStopping(patience=25) pass a new patience value, i.e. `patience=300` or use `patience=0` to disable EarlyStopping.

88 epochs completed in 2.945 hours.
Optimizer stripped from /kaggle/working/yolo_train/insect_yolo11m_opt/weights/last.pt, 40.5MB
Optimizer stripped from /kaggle/working/yolo_train/insect_yolo11m_opt/weights/best.pt, 40.5MB

Validating /kaggle/working/yolo_train/insect_yolo11m_opt/weights/best.pt...
Ultralytics 8.4.38 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
YOLO11m summary (fused): 126 layers, 20,037,742 parameters, 0 gradients, 67.7 GFLOPs
                 Class     Images  

libpng warning: iCCP: profile 'ICC Profile': 0h: PCS illuminant is not D50


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 27/27 2.4it/s 11.0s
                   all        851       2217      0.865      0.692      0.774       0.53
                   ant         96        478      0.827      0.565      0.699      0.465
             butterfly         83        288      0.817      0.767      0.826      0.645
             cockroach         77        184      0.756      0.446      0.558      0.347
             dragonfly         97        142       0.86      0.627      0.724      0.493
                   fly         97        178      0.932      0.876      0.896      0.653
           grasshopper         97        118      0.914      0.814      0.893      0.569
              honeybee         98        462      0.892       0.73       0.85      0.547
               ladybug         82        207      0.845      0.768      0.852      0.623
              mosquito         39         53      0.902      0.623     

# Validate

In [7]:
metrics = model.val(data="/kaggle/working/data_fixed.yaml", split="val")
print(metrics)

Ultralytics 8.4.38 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
YOLO11m summary (fused): 126 layers, 20,037,742 parameters, 0 gradients, 67.7 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 141.0±133.7 MB/s, size: 77.9 KB)
val: Scanning /kaggle/input/datasets/nguyenviet2709/data-yolo-v1/dataset_yolo/labels/val... 851 images, 0 backgrounds, 3 corrupt: 100% ━━━━━━━━━━━━ 854/854 458.8it/s 1.9s
val: /kaggle/input/datasets/nguyenviet2709/data-yolo-v1/dataset_yolo/images/val/ant__ant_0003.jpg: 2 duplicate labels removed
val: /kaggle/input/datasets/nguyenviet2709/data-yolo-v1/dataset_yolo/images/val/ant__ant_0006.jpg: 2 duplicate labels removed
val: /kaggle/input/datasets/nguyenviet2709/data-yolo-v1/dataset_yolo/images/val/ant__ant_0018.jpg: 2 duplicate labels removed
val: /kaggle/input/datasets/nguyenviet2709/data-yolo-v1/dataset_yolo/images/val/ant__ant_0029.jpg: 8 duplicate labels removed
val: /kaggle/input/datasets/nguyenviet2709/data-yolo-v1/dataset_yolo/image

libpng warning: iCCP: profile 'ICC Profile': 0h: PCS illuminant is not D50


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 54/54 2.3it/s 23.0s
                   all        851       2217      0.868      0.692      0.773      0.529
                   ant         96        478      0.829      0.569      0.696      0.463
             butterfly         83        288       0.82      0.767      0.827      0.646
             cockroach         77        184      0.755      0.446      0.557      0.347
             dragonfly         97        142      0.859       0.62      0.724      0.492
                   fly         97        178      0.937      0.876      0.896      0.652
           grasshopper         97        118      0.914      0.814      0.892      0.565
              honeybee         98        462      0.891      0.727      0.847      0.544
               ladybug         82        207      0.846      0.773       0.85      0.625
              mosquito         39         53      0.928      0.623     

# Test set

In [8]:
test_metrics = model.val(data="/kaggle/working/data_fixed.yaml", split="test")
print(test_metrics)

Ultralytics 8.4.38 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 11.6±14.1 MB/s, size: 99.1 KB)
val: Scanning /kaggle/input/datasets/nguyenviet2709/data-yolo-v1/dataset_yolo/labels/test... 438 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 438/438 146.2it/s 3.0s
val: /kaggle/input/datasets/nguyenviet2709/data-yolo-v1/dataset_yolo/images/test/ant__ant_0004.jpg: 24 duplicate labels removed
val: /kaggle/input/datasets/nguyenviet2709/data-yolo-v1/dataset_yolo/images/test/ant__ant_0010.jpg: 4 duplicate labels removed
val: /kaggle/input/datasets/nguyenviet2709/data-yolo-v1/dataset_yolo/images/test/ant__ant_0011.jpg: 12 duplicate labels removed
val: /kaggle/input/datasets/nguyenviet2709/data-yolo-v1/dataset_yolo/images/test/ant__ant_0040.jpeg: 2 duplicate labels removed
val: /kaggle/input/datasets/nguyenviet2709/data-yolo-v1/dataset_yolo/images/test/ant__ant_0044.jpeg: 4 duplicate labels removed
val: /kaggle/input/datase

# Predict thử

In [9]:
model.predict(
    source="/kaggle/working/dataset_yolo/images/test",
    imgsz=640,
    conf=0.25,
    save=True
)

FileNotFoundError: /kaggle/working/dataset_yolo/images/test does not exist

# Lấy model

In [ ]:
BEST = "/kaggle/working/yolo_train/insect_yolo11m_opt/weights/best.pt"
LAST = "/kaggle/working/yolo_train/insect_yolo11m_opt/weights/last.pt"

print("Best:", BEST)
print("Last:", LAST)

# Hiển thị results.png

In [ ]:
from IPython.display import Image, display

RESULTS_IMG = "/kaggle/working/yolo_train/insect_yolo11m_opt/results.png"

display(Image(filename=RESULTS_IMG))

# Hiển thị confusion matrix

In [ ]:
CONFUSION_IMG = "/kaggle/working/yolo_train/insect_yolo11m_opt/confusion_matrix.png"

display(Image(filename=CONFUSION_IMG))

# Predict trên test và lưu ảnh

In [ ]:
PREDICT_DIR = "/kaggle/working/predict_test"

results = model.predict(
    source="/kaggle/working/dataset_yolo/images/test",
    imgsz=640,
    conf=0.25,
    save=True,
    project=PREDICT_DIR,
    name="run1"
)

In [ ]:
import matplotlib.pyplot as plt
import cv2
import random
from pathlib import Path

PRED_IMG_DIR = Path(PREDICT_DIR) / "run1"

image_files = list(PRED_IMG_DIR.glob("*.jpg"))
sample_imgs = random.sample(image_files, min(6, len(image_files)))

plt.figure(figsize=(16, 10))

for i, img_path in enumerate(sample_imgs, 1):
    img = cv2.imread(str(img_path))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    plt.subplot(2, 3, i)
    plt.imshow(img)
    plt.title(img_path.name)
    plt.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
TEST_IMG = "/kaggle/working/dataset_yolo/images/test"

results = model.predict(
    source=TEST_IMG,
    imgsz=640,
    conf=0.25,
    save=False,
    show=False
)

for r in results[:5]:
    print("=" * 50)
    print("Image:", r.path)

    for box in r.boxes:
        cls_id = int(box.cls[0])
        conf = float(box.conf[0])
        print(f"Class: {cls_id}, Confidence: {conf:.2f}")

# Vẽ bbox trực tiếp (custom đẹp hơn)

In [ ]:
def draw_predictions(result, class_names):
    img = result.orig_img.copy()

    for box in result.boxes:
        cls_id = int(box.cls[0])
        conf = float(box.conf[0])
        x1, y1, x2, y2 = map(int, box.xyxy[0])

        label = f"{class_names[cls_id]} {conf:.2f}"

        cv2.rectangle(img, (x1, y1), (x2, y2), (255, 0, 0), 2)
        cv2.putText(img, label, (x1, max(20, y1 - 5)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 0, 0), 2)

    return img

In [ ]:
CLASS_NAMES = model.names

results = model.predict(
    source="/kaggle/working/dataset_yolo/images/test",
    imgsz=640,
    conf=0.25,
    save=False
)

plt.figure(figsize=(16, 10))

for i, r in enumerate(results[:6], 1):
    img = draw_predictions(r, CLASS_NAMES)

    plt.subplot(2, 3, i)
    plt.imshow(img)
    plt.title(Path(r.path).name)
    plt.axis("off")

plt.tight_layout()
plt.show()